In [1]:
import pandas as pd 
import numpy as np
import plotly.express as px 
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("insurance.csv")
print(df.head())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [4]:
length = len(df)

In [5]:
def findind_missing_pct(col):
    missing_pct = (df[col].isnull().sum()/length)*100
    return missing_pct

In [6]:
findind_missing_pct('age')

0.0

In [7]:
missing = [ f"{col} : {findind_missing_pct(col)}"  for col in df.columns]
missing

['age : 0.0',
 'sex : 0.0',
 'bmi : 0.0',
 'children : 0.0',
 'smoker : 0.0',
 'region : 0.0',
 'charges : 0.0']

In [8]:
df["charges"].describe()

count     1338.000000
mean     13270.422265
std      12110.011237
min       1121.873900
25%       4740.287150
50%       9382.033000
75%      16639.912515
max      63770.428010
Name: charges, dtype: float64

In [9]:
df["charges"].skew()

1.5158796580240388

Because skewness above +1.0 is considered high/strong skew, meaning the distribution has a long right tail with large extreme values.

In [10]:
fig = px.histogram(df, x="charges", nbins=50, title="Distribution of Insurance Charges")
fig.update_traces(marker=dict(line = dict(color ="black", width = 1)))
fig.show()

In [11]:
df["Log_charges"] = np.log(df["charges"])
df.head()

,age,sex,bmi,children,smoker,region,charges,Log_charges
0,19,female,27.900,0,yes,southwest,16884.92400,9.734176
1,18,male,33.770,1,no,southeast,1725.55230,7.453302
2,28,male,33.000,3,no,southeast,4449.46200,8.400538
3,33,male,22.705,0,no,northwest,21984.47061,9.998092
4,32,male,28.880,0,no,northwest,3866.85520,8.260197


In [12]:
fig = px.histogram(df, x="Log_charges", nbins=50, title="Distribution of Log Transformed Insurance Charges")
fig.update_traces(marker=dict(line = dict(color ="black", width = 1)))
fig.show()

In [13]:
print(df[["age","bmi","children"]].describe())
print(df[["age","bmi","children"]].skew())

               age          bmi     children
count  1338.000000  1338.000000  1338.000000
mean     39.207025    30.663397     1.094918
std      14.049960     6.098187     1.205493
min      18.000000    15.960000     0.000000
25%      27.000000    26.296250     0.000000
50%      39.000000    30.400000     1.000000
75%      51.000000    34.693750     2.000000
max      64.000000    53.130000     5.000000
age         0.055673
bmi         0.284047
children    0.938380
dtype: float64


In [14]:
outlier_detection = ["age","bmi","children"]

for col in outlier_detection:
    fig = px.box(df, y=col, title=f"Box plot of {col}")
    fig.update_traces(marker=dict(color="darkblue", line=dict(color="black", width=1)))
    fig.show()

In [15]:
for  col in outlier_detection:
    fig = px.histogram(df, x=col, nbins=50, title=f"Distribution of {col}")
    fig.update_traces(marker=dict(line = dict(color ="black", width = 1)))
    fig.show()

In [16]:
print(df[["sex","smoker","region"]].nunique())
cat_col = ["sex","smoker","region"]
for col in cat_col:
    print(df[col].value_counts())

sex       2
smoker    2
region    4
dtype: int64
sex
male      676
female    662
Name: count, dtype: int64
smoker
no     1064
yes     274
Name: count, dtype: int64
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64


In [17]:
df.head()

,age,sex,bmi,children,smoker,region,charges,Log_charges
0,19,female,27.900,0,yes,southwest,16884.92400,9.734176
1,18,male,33.770,1,no,southeast,1725.55230,7.453302
2,28,male,33.000,3,no,southeast,4449.46200,8.400538
3,33,male,22.705,0,no,northwest,21984.47061,9.998092
4,32,male,28.880,0,no,northwest,3866.85520,8.260197


In [18]:
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [19]:
cat_col

['sex', 'smoker', 'region']

In [20]:
preprocessor = ColumnTransformer([("cat", OneHotEncoder(drop="first", handle_unknown="ignore"),cat_col)],remainder="passthrough")

In [21]:
from sklearn.linear_model import LinearRegression
pipeline = Pipeline([
    ("prep", preprocessor),
    ("lr", LinearRegression())
])

In [22]:
x = df.drop(columns=["charges","Log_charges"])
y = df["Log_charges"]

pipeline.fit(x,y)

,steps,"[('prep', ...), ('lr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [23]:
y_pred = pipeline.predict(x)

residuals = y - y_pred

In [24]:
print(residuals.mean())
print(residuals.std())
print(residuals.skew())


1.5148155110281657e-15
0.44295276896150665
1.6804041706728


In [25]:
fig = px.histogram(residuals, x=residuals, nbins=50, title="Distribution of Residuals")
fig.update_traces(marker=dict(line = dict(color ="black", width = 1)))
fig.show()

In [26]:
fig = px.scatter(x=y_pred, y=residuals, title="Residuals vs Predicted Values", labels={"x":"Predicted Values", "y":"Residuals"})
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

In [27]:
import pandas as pd

# Create a DataFrame with both actual and predicted values
plot_df = pd.DataFrame({
    'Values': list(y) + list(y_pred),
    'Type': ['Actual'] * len(y) + ['Predicted'] * len(y_pred),
    'Index': list(range(len(y))) + list(range(len(y_pred)))
})

# Create scatter plot with color differentiation
fig = px.scatter(
    plot_df, 
    x='Index', 
    y='Values', 
    color='Type',
    title="Actual vs Predicted Values",
    color_discrete_map={'Actual': 'blue', 'Predicted': 'orange'}
)
fig.show()

In [28]:
df["smoker_yes"] = (df["smoker"] == "yes").astype(int)
df.head()

,age,sex,bmi,children,smoker,region,charges,Log_charges,smoker_yes
0,19,female,27.900,0,yes,southwest,16884.92400,9.734176,1
1,18,male,33.770,1,no,southeast,1725.55230,7.453302,0
2,28,male,33.000,3,no,southeast,4449.46200,8.400538,0
3,33,male,22.705,0,no,northwest,21984.47061,9.998092,0
4,32,male,28.880,0,no,northwest,3866.85520,8.260197,0


In [29]:
df["bmi_smoker"] = df["bmi"] * df["smoker_yes"]
df["age_smoker"] = df["age"] * df["smoker_yes"]
df["bmi_age"]  = df["bmi"] * df["age"]
df.head()

,age,sex,bmi,children,smoker,region,charges,Log_charges,smoker_yes,bmi_smoker,age_smoker,bmi_age
0,19,female,27.900,0,yes,southwest,16884.92400,9.734176,1,27.9,19,530.100
1,18,male,33.770,1,no,southeast,1725.55230,7.453302,0,0.0,0,607.860
2,28,male,33.000,3,no,southeast,4449.46200,8.400538,0,0.0,0,924.000
3,33,male,22.705,0,no,northwest,21984.47061,9.998092,0,0.0,0,749.265
4,32,male,28.880,0,no,northwest,3866.85520,8.260197,0,0.0,0,924.160


In [30]:
new_x = df.drop(columns=["charges", "Log_charges"])
new_y = df["Log_charges"]

pipeline.fit(new_x,new_y)

,steps,"[('prep', ...), ('lr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [31]:
new_y_pred = pipeline.predict(new_x)

new_residuals = new_y - new_y_pred

In [32]:
print(new_residuals.mean())
print(new_residuals.std())
print(new_residuals.skew())


1.1331244861196665e-15
0.3847384048388007
2.972137316755025


In [33]:
fig = px.scatter(x=new_y_pred, y=new_residuals, title="Residuals vs Predicted Values", labels={"x":"Predicted Values", "y":"Residuals"})
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

In [34]:
import pandas as pd

# Create a DataFrame with both actual and predicted values
plot_df = pd.DataFrame({
    'Values': list(new_y) + list(new_y_pred),
    'Type': ['Actual'] * len(new_y) + ['Predicted'] * len(new_y_pred),
    'Index': list(range(len(new_y))) + list(range(len(new_y_pred)))
})

# Create scatter plot with color differentiation
fig = px.scatter(
    plot_df, 
    x='Index', 
    y='Values', 
    color='Type',
    title="Actual vs Predicted Values",
    color_discrete_map={'Actual': 'blue', 'Predicted': 'orange'}
)
fig.show()

In [35]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_cols = ["sex", "smoker", "region"]
numeric_cols = ["age", "bmi", "children"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)
], remainder="passthrough")

rf_model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=10,               # limits complexity
        min_samples_split=20,       # smoother splits
        min_samples_leaf=10,        # avoids tiny leaves
        max_features='sqrt',        # stronger regularization
        bootstrap=True,
        random_state=42
    ))
])

X = df.drop(columns=["charges", "Log_charges"])
y = df["charges"]    # Use ORIGINAL target for tree models!

rf_model.fit(X, y)
y_pred = rf_model.predict(X)
residuals = y - y_pred


In [36]:
print(residuals.mean())
print(residuals.std())
print(residuals.skew())


-6.401435076896653
4070.600375814056
3.173023184748312


In [37]:
fig = px.scatter(x=y_pred, y=residuals, title="Residuals vs Predicted Values", labels={"x":"Predicted Values", "y":"Residuals"})
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

import pandas as pd

# Create a DataFrame with both actual and predicted values
plot_df = pd.DataFrame({
    'Values': list(y) + list(y_pred),
    'Type': ['Actual'] * len(y) + ['Predicted'] * len(y_pred),
    'Index': list(range(len(y))) + list(range(len(y_pred)))
})

# Create scatter plot with color differentiation
fig = px.scatter(
    plot_df, 
    x='Index', 
    y='Values', 
    color='Type',
    title="Actual vs Predicted Values",
    color_discrete_map={'Actual': 'blue', 'Predicted': 'orange'}
)
fig.show()

In [38]:
from sklearn.model_selection import train_test_split

categorical_cols = ["sex", "smoker", "region"]
numeric_cols = ["age", "bmi", "children"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)
], remainder="passthrough")

X = df.drop(columns=["charges", "Log_charges"])
y = df["charges"]          # RAW charges for ML & Tweedie

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [39]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rf_model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("RF RMSE:", rf_rmse)
print("RF R2:", rf_r2)


RF RMSE: 4484.719321091663
RF R2: 0.8704484912970859


In [40]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_model = Pipeline([
    ("prep", preprocessor),
    ("gbr", GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.9,
        random_state=42
    ))
])

gbr_model.fit(X_train, y_train)
gbr_pred = gbr_model.predict(X_test)

gbr_rmse = np.sqrt(mean_squared_error(y_test, gbr_pred))
gbr_r2 = r2_score(y_test, gbr_pred)

print("GBR RMSE:", gbr_rmse)
print("GBR R2:", gbr_r2)


GBR RMSE: 4516.109744441694
GBR R2: 0.8686285741090133


In [41]:
from sklearn.linear_model import TweedieRegressor
from sklearn.preprocessing import StandardScaler

tweedie_model = Pipeline([
    ("prep", preprocessor),
    ("scale", StandardScaler(with_mean=False)),  # required for GLM stability
    ("glm", TweedieRegressor(
        power=1.5,          # between Gamma (p=2) & Poisson (p=1)
        alpha=0.1,          # L2 regularization
        max_iter=5000
    ))
])

tweedie_model.fit(X_train, y_train)
tw_pred = tweedie_model.predict(X_test)

tw_rmse = np.sqrt(mean_squared_error(y_test, tw_pred))
tw_r2 = r2_score(y_test, tw_pred)

print("Tweedie RMSE:", tw_rmse)
print("Tweedie R2:", tw_r2)


Tweedie RMSE: 5078.071870960602
Tweedie R2: 0.8339000024099538


In [42]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import TweedieRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols = ["sex", "smoker", "region"]
numeric_cols = ["age", "bmi", "children"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols)
])

estimators = [
    ('rf', RandomForestRegressor(n_estimators=300,
        max_depth=10,               # limits complexity
        min_samples_split=20,       # smoother splits
        min_samples_leaf=10,        # avoids tiny leaves
        max_features='sqrt',        # stronger regularization
        bootstrap=True,
        random_state=42)),
    ('gbr', GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=4, subsample=0.9, random_state=42
    )),
    ('tweedie', TweedieRegressor(power=1.5, alpha=0.1, max_iter=5000))
]

stack_model = Pipeline([
    ("prep", preprocessor),
    ("stack", StackingRegressor(
        estimators = estimators,
        final_estimator = Ridge(alpha=1.0)
    ))
])

stack_model.fit(X_train, y_train)
stack_pred = stack_model.predict(X_test)


In [43]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

stack_rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
stack_r2 = r2_score(y_test, stack_pred)

print("Stack RMSE:", stack_rmse)
print("Stack R2:", stack_r2)


Stack RMSE: 4437.8903513958985
Stack R2: 0.873139892469547


In [50]:
fig = px.histogram(X_train, x = "age", title="age distribution", nbins = 25)
fig.update_traces(marker = dict(color = "red", line = dict(color = "black",width = 1)))
fig.show()

In [51]:
fig = px.histogram(X_train, x = "bmi", title="bmi distribution", nbins = 25)
fig.update_traces(marker = dict(color = "orange", line = dict(color = "black",width = 1)))
fig.show()

In [44]:
import torch 
print(torch.__version__)

2.7.0.dev20250310+cu124


In [52]:
import torch.nn as nn
import torch.optim as optim 
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer

In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols = ["sex", "smoker", "region"]
numeric_cols = ["age", "bmi", "children"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols)
])


In [54]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["charges", "Log_charges"])
y = df["Log_charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [55]:
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre  = preprocessor.transform(X_test)

In [56]:
X_train_tensor = torch.tensor(X_train_pre, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_pre, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [57]:
import torch.nn as nn

class InsuranceNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.model(x)
        

input_dim = X_train_pre.shape[1]
model = InsuranceNN(input_dim)
model


InsuranceNN(
  (model): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.15, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): ReLU()
    (10): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [58]:
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20, eta_min=1e-5
)

In [59]:
import numpy as np

best_loss = float("inf")
patience = 20
patience_counter = 0

epochs = 200

for epoch in range(epochs):
    model.train()
    running_loss = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    scheduler.step()
    train_loss = running_loss / len(train_loader)

    # Early stopping check
    if train_loss < best_loss:
        best_loss = train_loss
        patience_counter = 0
        best_state = model.state_dict()
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print("Early stopping triggered.")
        break

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {train_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

model.load_state_dict(best_state)


Epoch 10, Loss: 1.1777, LR: 0.000505
Epoch 20, Loss: 1.0349, LR: 0.000010
Epoch 30, Loss: 0.9461, LR: 0.000505
Epoch 40, Loss: 1.0500, LR: 0.001000
Epoch 50, Loss: 0.8326, LR: 0.000505
Epoch 60, Loss: 0.8216, LR: 0.000010
Epoch 70, Loss: 0.7706, LR: 0.000505
Early stopping triggered.


<All keys matched successfully>

In [61]:
from sklearn.metrics import mean_squared_error, r2_score

model.eval()
with torch.no_grad():
    preds = model(X_test_tensor).numpy()

mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, preds)

print("NN RMSE:", rmse)
print("NN R2:", r2)


NN RMSE: 0.5327355539551387
NN R2: 0.6843573181372788
